In [1]:
using LowLevelFEM, LinearAlgebra

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 1 for different dependency version already loaded, 9 for different Julia build configuration)
Precompiling packages...
   1.5 s  ✓ CommonWorldInvalidations
   1.8 s  ✓ Adapt
   1.7 s  ✓ ChunkCodecCore
   2.1 s  ✓ SciMLPublic
   2.5 s  ✓ OrderedCollections
   2.3 s  ✓ CpuId
   4.0 s  ✓ MPIPreferences
   4.3 s  ✓ ThreadingUtilities
   3.7 s  ✓ Libmount_jll
   5.6 s  ✓ METIS_jll
   2.4 s  ✓ libaec_jll
   4.4 s  ✓ Libffi_jll
   3.1 s  ✓ aws_c_common_jll
   5.0 s  ✓ Libuuid_jll
   3.7 s  ✓ Xorg_libpciaccess_jll
   4.9 s  ✓ libpng_jll
   4.8 s  ✓ Bzip2_jll
   4.8 s  ✓ LLVMOpenMP_jll
   4.8 s  ✓ s2n_tls_jll
   4.4 s  ✓ Xorg_libXau_jll
   6.9 s  ✓ Libiconv_jll
   7.2 s  ✓ Zstd_jll
   6.5 s  ✓ Xorg_libXdmcp_jll
   4.4 s  ✓ dlfcn_win32_jll
   4.7 s  ✓ Xorg_xtrans_jll
   4.1 s  ✓ JpegTurbo_jll
   6.7 s  ✓ Expat_jll
   1.5 s  ✓ ChunkCodecLibZlib
   3.8 s  ✓ ArrayInterface
   3.6 s  ✓ Adapt → AdaptSparseArra

In [2]:
structured_rect_mesh(lx=10)
#openPreProcessor()
mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u)
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U), Ω="body")
f = ∫(U ⋅ [100,-100], Ω="body")
bc = BoundaryCondition("left", ux=1, uy=1) # <====
per = MultiPointConstraint(master="left", slave="right", ux=false, uy=true)
u = solveField(Symmetric(K), f, support=[bc], mpc=[per])
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();

XOpenIM() failed
Fontconfig warning: using without calling FcInit()


In [9]:
structured_rect_mesh(lx=10)
#openPreProcessor()
mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u)
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U), Ω="body")
Kc = ∫(U ⋅ 1e4 ⋅ U, Γ="left") # <====
f = ∫(U ⋅ [100,-100], Ω="body")
per = MultiPointConstraint(master="left", slave="right", ux=false, uy=true)
u = solveField(Symmetric(K + Kc), f, mpc=[per])
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();

In [57]:
structured_rect_mesh(lx=10)
#openPreProcessor()
mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u)
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U), Ω="body")
Kc = ∫(U ⋅ 1e4 ⋅ U, Γ="left")
s = ScalarField(U, "body", (x,y,z)->x/10)
f = ∫(U ⋅ [100,-100s], Ω="body")
e1 = VectorField(U, "left", [1,0,0]) # <====
cs = CoordinateSystem(e1, handedness=:left) # <====
per = MultiPointConstraint(master="left", slave="right", ux=true, uy=true)
u = solveField(Symmetric(K + Kc), f, mpc=[per], coordSys=[cs])
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();

In [51]:
structured_rect_mesh(lx=10, rightRemote=true)
#openPreProcessor()
mat = Material("body")
rem = Material("rightremote")
U = Field([mat,rem], type=:VectorField, dim=2, fieldName=:u)
Φ = Field([mat,rem], type=:ScalarField, dim=2, fieldName=:φ)
Ku = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U), Ω="body")
Ku += 0*∫(U ⋅ 1e4 ⋅ U, Γ="left")
mpc_U = MultiPointConstraint(master="rightremote", slave="right", field=U)
mpc_Φ = MultiPointConstraint(master="rightremote", slave="right", field=Φ)
R = rigidRotationMap(mpc_U, mpc_Φ)
Kuφ = Ku * R
Kφ = R' * Kuφ
K = SystemMatrix([ Ku Kuφ; Kuφ' Kφ])
fu = ∫(U ⋅ [1,0], Γ="rightremote")
fφ = ∫(Φ ⋅ 1000, Γ="rightremote")
f = SystemVector([fu, fφ])
bc = BoundaryCondition("left", ux=0, uy=0, field=U)
e1 = VectorField(U, "left", [1,0,0])
cs = CoordinateSystem(e1, handedness=:right)
per = MultiPointConstraint(master="left", slave="right", ux=true, uy=true, field=U)
u0, φ = solveField(Symmetric(K), f, support=[bc], mpc=[per, mpc_U, mpc_Φ], coordSys=[cs])
u = u0 + R * φ
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();

In [50]:
structured_rect_mesh(lx=10, rightRemote=true)
#openPreProcessor()
mat = Material("body")
rem = Material("rightremote")
U = Field([mat,rem], type=:VectorField, dim=2, fieldName=:u)
Φ = Field([mat,rem], type=:ScalarField, dim=2, fieldName=:φ)
Ku = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U), Ω="body")
Ku += 0*∫(U ⋅ 1e4 ⋅ U, Γ="left")
mpc_U = MultiPointConstraint(master="rightremote", slave="right", field=U)
mpc_Φ = MultiPointConstraint(master="rightremote", slave="right", field=Φ)
R = rigidRotationMap(mpc_U, mpc_Φ)
Kuφ = Ku * R
Kφ = R' * Kuφ
K = SystemMatrix([ Ku Kuφ; Kuφ' Kφ])
fu = ∫(U ⋅ [1,0], Γ="rightremote")
fφ = ∫(Φ ⋅ 1000, Γ="rightremote")
f = SystemVector([fu, fφ])
e1 = VectorField(U, "left", [-1,0,0])
cs = CoordinateSystem(e1, handedness=:right)
per = MultiPointConstraint(master="left", slave="right", ux=true, uy=true, field=U)
u0, φ = solveField(Symmetric(K), f, mpc=[per, mpc_U, mpc_Φ], coordSys=[cs])
u = u0 + R * φ
showDoFResults(u, name="u", factor=1, visible=true)
showDoFResults(u0, name="u0", factor=1)
openPostProcessor();

In [8]:
structured_box_mesh(lx=10, rightRemote=true)
#openPreProcessor()
mat = Material("body")
rem = Material("rightremote")
U = Field([mat,rem], type=:VectorField, dim=3, fieldName=:u)
Φ = Field([mat,rem], type=:VectorField, dim=3, fieldName=:φ)
Ku = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U), Ω="body")
Ku += 0*∫(U ⋅ 1e4 ⋅ U, Γ="left")
mpc_U = MultiPointConstraint(master="rightremote", slave="right", field=U)
mpc_Φ = MultiPointConstraint(master="rightremote", slave="right", field=Φ)
R = rigidRotationMap(mpc_U, mpc_Φ)
Kuφ = Ku * R
Kφ = R' * Kuφ
K = SystemMatrix([ Ku Kuφ; Kuφ' Kφ])
fu = ∫(U ⋅ [0,0,0], Γ="rightremote")
fφ = ∫(Φ ⋅ [0,0,0], Γ="rightremote")
f = SystemVector([fu, fφ])
bcU = BoundaryCondition("rightremote", field=U, ux=0, uy=0, uz=0)
bcΦ = BoundaryCondition("rightremote", field=Φ, φx=0.5, φy=0, φz=0)
e1 = VectorField(U, "left", [-1,0,0]) # <===
e2 = VectorField(U, "left", [0,-1,0])
cs = CoordinateSystem(e1, e2, handedness=:left)
per = MultiPointConstraint(master="left", slave="right", ux=true, uy=true, uz=true, field=U)
u0, φ = solveField(Symmetric(K), f, support=[bcU, bcΦ], mpc=[per, mpc_U, mpc_Φ], coordSys=[cs])
u = u0 + R * φ
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();

In [17]:
structured_box_mesh(lx=10, leftRemote=true, rightRemote=true)
#openPreProcessor()
mat = Material("body")
remL = Material("leftremote")
remR = Material("rightremote")
U = Field([mat,remL,remR], type=:VectorField, dim=3, fieldName=:u)
Φ = Field([mat,remL,remR], type=:VectorField, dim=3, fieldName=:φ)
ε = SymGrad
Ku = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U), Ω="body")
Ku += 0*∫(U ⋅ 1e4 ⋅ U, Γ="left")
mpc_Ul = MultiPointConstraint(master="leftremote", slave="left", field=U)
mpc_Φl = MultiPointConstraint(master="leftremote", slave="left", field=Φ)
mpc_Ur = MultiPointConstraint(master="rightremote", slave="right", field=U)
mpc_Φr = MultiPointConstraint(master="rightremote", slave="right", field=Φ)
R = rigidRotationMap(mpc_Ul, mpc_Φl)
R += rigidRotationMap(mpc_Ur, mpc_Φr)
Kuφ = Ku * R
Kφ = R' * Kuφ
K = SystemMatrix([ Ku Kuφ; Kuφ' Kφ])
fu = ∫(U ⋅ [0,0,0], Γ="rightremote")
fφ = ∫(Φ ⋅ [1,0,0], Γ="rightremote")
f = SystemVector([fu, fφ])
e1 = VectorField(U, "left", [-1,0,0]) # <===
e2 = VectorField(U, "left", [0,-1,0])
cs = CoordinateSystem(e1, e2, handedness=:left)
perU = MultiPointConstraint(master="left", slave="right", ux=true, uy=true, uz=true, field=U)
perΦ = MultiPointConstraint(master="left", slave="right", φx=true, φy=true, φz=true, field=Φ)
u0, φ = solveField(Symmetric(K), f, mpc=[perU, perΦ, mpc_Ul, mpc_Φl, mpc_Ur, mpc_Φr], coordSys=[cs])
u = u0 + R * φ
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();

In [14]:
structured_box_mesh(lx=10)
#openPreProcessor()
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)
ε = SymGrad
K = ∫(ε(U) ⋅ D(:Solid, mat) ⋅ ε(U), Ω="body")
Kcl = ∫(U ⋅ 1e4 ⋅ U, Γ="left")
Kcr = ∫(U ⋅ 1e4 ⋅ U, Γ="right")
uy = ScalarField(U, "right", (x,y,z)->z-0.5)
uz = ScalarField(U, "right", (x,y,z)->-(y-0.5))
u0 = VectorField([0, uy, uz])
f = ∫(U ⋅ [0,0,0], Γ="right")
e1 = VectorField(U, "left", [-1,0,0]) # <===
e2 = VectorField(U, "left", [0,-1,0])
cs = CoordinateSystem(e1, e2, handedness=:left)
per = MultiPointConstraint(master="left", slave="right", ux=true, uy=true, uz=true, field=U)
u = solveField(Symmetric(K), f-Kcr*u0, mpc=[per], coordSys=[cs])
showDoFResults(u, name="u", factor=1, visible=true)

ε = (u ∘ ∇ + ∇ ∘ u) / 2
E = mat.E
ν = mat.ν
I = unitTensor(ε)
σ = E / (1+ν) * (ε + ν / (1-2ν) * trace(ε) * I)

ex = VectorField(U, "body", [1,0,0])
showElementResults(σ * ex, name="σx")
openPostProcessor();